# Why the QELM state readout decays with p N T

Issue #305, after #268 (PR #274) and #285 (PR #302). On a fixed monitored Haar circuit, the ridge on the final-state features `<P_k>`, `<P_i P_j>` reads the sector of the reference qubit `R` only for a few measurements, `n = p N T` of order 4.

A linear readout of features `x` sees two things: the class means `mu_+-` and the within-class covariance `Sigma`. With `s = mu_+ - mu_-`, the best linear readout has, for Gaussian features, `acc = Phi(sqrt(s^T Sigma^-1 s) / 2)`.

- **Signal (exact).** `mu_+- = Tr(O rho_+-)`, with `rho_+-` the record-averaged final state of each sector: each round, the gates, then on each site the Z-dephasing of probability `p`. Dephasing site `k` multiplies the Pauli components of `Delta = rho_+ - rho_-` that have `X` or `Y` on `k` by `1 - p`. For a scrambled `Delta`, half the weight is there on every site, so `||Delta||_HS^2` loses a factor `1 - p` per site and round: `||Delta||_HS^2 = exp(-p V_eff)`, `V_eff ≈ N T`.
- **Noise (sampled).** At `p = 0` each sector gives one state, so `Sigma = 0`. It grows with `p`, like `p^beta`.
- **Accuracy.** `SNR^2 ∝ exp(-gamma n) / p^beta`: the midpoint follows `gamma n_0 = C - beta ln p_0`, a logarithmic correction to `n_0 = const`.

The notebook runs `experiments/pnt_decay/pnt_decay.sh` (or reads its latest sweep) and draws the six figures of `plot_pnt_decay.py`.

In [ ]:
import os
from pathlib import Path

# Jupyter starts in the notebook's folder. Move to the repository root.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"cwd: {Path.cwd()}")

## Parameters

Each value can be changed here or through the environment variable of the same name (`tests/test_pnt_decay_notebook.py` runs a small sweep this way). The default grid takes about 2.5 hours on 4 cores for one encoding and 8 circuits; on the cluster, add `12:6,12,24,48` to `SIZES` and raise `MAX_JOBS`. `RUN = False` reads back the latest sweep of `outputs/pnt_decay/`.

In [ ]:
ENCODINGS = os.environ.get("ENCODINGS", "charge bell")
SIZES = os.environ.get("SIZES", "4:2,4,8,16 6:3,6,12,24 8:4,8,16,32 10:5,10,20,40")
SEEDS = os.environ.get("SEEDS", "42 43 44 45 46 47 48 49")
PER_P = os.environ.get("PER_P", "3000")
PNT_VALUES = os.environ.get("PNT_VALUES", "")  # empty: the grid of the config
MAX_JOBS = os.environ.get("MAX_JOBS", "4")
OUT_ROOT = Path(os.environ.get("OUT_ROOT", "outputs"))
RUN = os.environ.get("RUN", "0") == "1"

## Launch the sweep

One run per encoding, `(N, T)` and seed (`experiments/pnt_decay/pnt_decay.sh`).

In [ ]:
from QELM.utils import latest_sweep, new_sweep_dir, run_with_progress

EXPERIMENT = "pnt_decay"
n_runs = (
    len(ENCODINGS.split())
    * len(SEEDS.split())
    * sum(len(size.split(":")[1].split(",")) for size in SIZES.split())
)
if RUN:
    sweep_dir = new_sweep_dir(OUT_ROOT, EXPERIMENT)
    env = {
        **os.environ,
        "ENCODINGS": ENCODINGS,
        "SIZES": SIZES,
        "SEEDS": SEEDS,
        "PER_P": PER_P,
        "MAX_JOBS": MAX_JOBS,
        "OUT_DIR": str(sweep_dir),
    }
    if PNT_VALUES:
        env["PNT_VALUES"] = PNT_VALUES
    run_with_progress(
        ["bash", "experiments/pnt_decay/pnt_decay.sh"],
        total=n_runs,
        env=env,
        show=("encoding", "N", "T", "seed"),
    )
else:
    sweep_dir = latest_sweep(OUT_ROOT / EXPERIMENT)
print(f"sweep: {sweep_dir}")

## Read back the runs and summarize

Per encoding: the decay rate `gamma` of the feature signal, the noise exponent `beta`, and per `(N, T)` the volume `V_eff / (N T)` and the midpoints `n_0 = p_0 N T` of the three accuracies.

In [ ]:
import importlib.util

import matplotlib.pyplot as plt

from QELM.utils.plots import save_figure

spec = importlib.util.spec_from_file_location(
    "plot_pnt_decay", "experiments/pnt_decay/plot_pnt_decay.py"
)
plot = importlib.util.module_from_spec(spec)
spec.loader.exec_module(plot)

# VOLUMES: sweeps run with SAMPLE=0 that add V_feat to older runs.
volumes = [Path(v) for v in os.environ.get("VOLUMES", "").split()]
rows = plot.load_rows([sweep_dir], volumes)
summary = plot.summarize(rows)
for enc, info in summary["encodings"].items():
    print(
        f"{enc}: gamma = {info['signal_rate']:.3f}, "
        f"noise ~ p^{info['noise_fit']['beta']:.2f}"
    )
    print(
        f"{'N':>3} {'T':>3} {'Veff/NT':>8} {'Vfeat/NT':>9}"
        f" {'n0 ridge':>9} {'oracle':>7} {'gauss':>7}"
    )
    for e in info["sizes"].values():
        print(
            f"{e['N']:>3} {e['T']:>3} {e['V_eff_over_NT']:8.3f}"
            f" {e['V_feat_over_NT']:9.3f}"
            f" {e['n0_ridge']:9.2f} {e['n0_oracle']:7.2f} {e['n0_gauss']:7.2f}"
        )

## Figures

1. `signal_decay`: exact `||Delta||_HS^2` against `n` and `p V_eff`, the signal of the features against `n` and `p V_feat`, the trace distance.
2. `noise`: the within-class variance against `p`, and `V_eff / (N T)`, `V_feat / (N T)` against `N / T` with the light-cone line.
3. `accuracy_vs_pNT`: ridge, oracle and Gaussian prediction against `n`.
4. `collapse`: the ridge against `p N T` and against `p V_feat`, with the spread of the curves.
5. `accuracy_vs_snr`: accuracy against `sqrt(SNR^2)` with `Phi(x / 2)`.
6. `accuracy_per_K`: the ridge at a fixed number of measurements `K`.
7. `midpoints`: `n_0` against `N T` and against `-ln p_0`, and `p_0 V_feat` against `-ln p_0`.

In [ ]:
for name in plot.FIGURES:
    fig, table = plot.DRAWERS[name](rows, summary)
    figure_path = save_figure(fig, sweep_dir, name, data=table)
    plt.show()